In [0]:
WITH latest_config_per_option AS (
  SELECT 
    tour_option_id,
    pricing_id,
    currency_iso_code,
    details_deserialized,
    update_timestamp,
    
    ROW_NUMBER() OVER (
      PARTITION BY tour_option_id, pricing_id
      ORDER BY update_timestamp DESC
    ) as rn
    
  FROM db_mirror_dbz.inventory_generated__available_time_slot
  WHERE details_deserialized IS NOT NULL
),

pricing_config_exploded AS (
  SELECT 
    tour_option_id,
    pricing_id,
    currency_iso_code,
    
    from_json(
      details_deserialized, 
      'struct<pricingCategories:struct<individual:array<struct<ticketCategory:string,minAge:int,maxAge:int,resolvedCommissionRate:decimal(10,3),independentlyBookable:boolean,bookingType:string,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,group:array<struct<resolvedCommissionRate:decimal(10,3),isAdditionalPaxForGroup:boolean,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,addons:array<struct<addonId:int,resolvedCommissionRate:decimal(10,3),prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>>,group:array<struct<resolvedCommissionRate:decimal(10,3),isAdditionalPaxForGroup:boolean,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,addons:array<struct<addonId:int,resolvedCommissionRate:decimal(10,3),prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>>'
    ) as parsed_data,
    
    update_timestamp as config_snapshot_timestamp
    
  FROM latest_config_per_option
  WHERE rn = 1
),

normalized_pricing AS (
  SELECT
    tour_option_id,
    pricing_id,
    currency_iso_code,
    
    COALESCE(parsed_data.pricingCategories.individual, ARRAY()) as individual_categories,
    
    COALESCE(
      parsed_data.pricingCategories.group,
      parsed_data.group
    ) as group_categories_raw,
    
    COALESCE(
      parsed_data.pricingCategories.addons,
      parsed_data.addons
    ) as addons_raw,
    
    config_snapshot_timestamp
    
  FROM pricing_config_exploded
)

SELECT 
  tour_option_id,
  pricing_id,
  currency_iso_code,
  
  TRANSFORM(
    individual_categories,
    ind -> STRUCT(
      ind.ticketCategory as ticket_category,
      ind.minAge as min_age,
      ind.maxAge as max_age,
      ind.resolvedCommissionRate as commission_rate,
      SIZE(ind.prices) as num_price_tiers,
      ind.prices as price_tiers
    )
  ) as individual_categories,
  
  TRANSFORM(
    group_categories_raw,
    grp -> STRUCT(
      grp.resolvedCommissionRate as commission_rate,
      grp.isAdditionalPaxForGroup as is_additional_pax,
      SIZE(grp.prices) as num_price_tiers,
      grp.prices as price_tiers
    )
  ) as group_categories,
  
  TRANSFORM(
    addons_raw,
    addon -> STRUCT(
      addon.addonId as addon_id,
      addon.resolvedCommissionRate as commission_rate,
      SIZE(addon.prices) as num_price_tiers,
      addon.prices as price_tiers
    )
  ) as addons,
  
  -- COUNT OF CATEGORIES/GROUPS/ADDONS
  SIZE(individual_categories) as num_individual_categories,
  SIZE(group_categories_raw) as num_group_categories,
  SIZE(addons_raw) as num_addons,
  
  -- TOTAL NUMBER OF PRICE TIERS
  AGGREGATE(individual_categories, 0, (acc, ind) -> acc + SIZE(ind.prices)) as total_individual_price_tiers,
  AGGREGATE(group_categories_raw, 0, (acc, grp) -> acc + SIZE(grp.prices)) as total_group_price_tiers,
  AGGREGATE(addons_raw, 0, (acc, addon) -> acc + SIZE(addon.prices)) as total_addon_price_tiers,
  
  -- HAS VOLUME PRICING FLAGS
  CASE 
    WHEN SIZE(FILTER(individual_categories, ind -> SIZE(ind.prices) > 1)) > 0 
    THEN 1 ELSE 0 
  END as has_volume_pricing_individual,
  
  CASE 
    WHEN SIZE(FILTER(group_categories_raw, grp -> SIZE(grp.prices) > 1)) > 0 
    THEN 1 ELSE 0 
  END as has_volume_pricing_group,
  
  CASE 
    WHEN SIZE(FILTER(addons_raw, addon -> SIZE(addon.prices) > 1)) > 0 
    THEN 1 ELSE 0 
  END as has_volume_pricing_addons,
  
  config_snapshot_timestamp

FROM normalized_pricing;
